In [0]:
# src/transformation/silver_cleaning.py
from pyspark.sql import functions as F
from pyspark.sql import types as T
import re

# -------- CONFIG --------
RAW_GLOB = "/Volumes/bls_cew/raw/bronze_cew/*.csv"

SILVER_PATH = "/Volumes/bls_cew/silver/silver_cew/cleaned_data"

WRITE_FORMAT = "delta"   
WRITE_MODE = "overwrite"
PARTITION_COL = "year"

# -------- Helpers --------
def normalize_col(name: str) -> str:
    s = name.strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

# -------- READ --------
df_raw = spark.read.csv(RAW_GLOB, header=True, inferSchema=True)

# -------- RENAME COLS --------
for c in df_raw.columns:
    df_raw = df_raw.withColumnRenamed(c, normalize_col(c))

df = df_raw

# -------- DROP DISCLOSURE --------
to_drop = [c for c in ["disclosure_code", "lq_disclosure_code", "oty_disclosure_code"] if c in df.columns]
df = df.drop(*to_drop)

# -------- FILTER agglvl_code = 73 --------
if "agglvl_code" not in df.columns:
    raise ValueError("Column agglvl_code not found.")
df = df.filter(F.col("agglvl_code") == 73)

# -------- KEEP own_code + sector_type --------
if "own_code" not in df.columns:
    raise ValueError("Column own_code not found.")
df = df.withColumn(
    "sector_type",
    F.when(F.col("own_code") == 1, F.lit("Private"))
     .when(F.col("own_code").isin(2, 3, 5), F.lit("Public"))
     .otherwise(F.lit("Other"))
)


# Remove negative measures
measures = [
    "annual_avg_estabs","annual_avg_emplvl","total_annual_wages","taxable_annual_wages",
    "annual_contributions","annual_avg_wkly_wage","avg_annual_pay"
]
for c in measures:
    if c in df.columns:
        df = df.withColumn(c, F.when(F.col(c) < 0, F.lit(None)).otherwise(F.col(c)))

# -------- DEDUP --------
key_cols = [c for c in ["year","area_fips","industry_code","own_code","agglvl_code","size_code"] if c in df.columns]
if key_cols:
    df = df.dropDuplicates(key_cols)
else:
    df = df.dropDuplicates()

# -------- VALIDATION  --------
rows = df.count()
print("Rows after filter & cleaning:", rows)
df.groupBy("sector_type").count().show()
df.groupBy("year").count().orderBy("year").show(50, truncate=False)

# # -------- WRITE SILVER --------
(df.write.format(WRITE_FORMAT)
   .mode("overwrite")
   .save(SILVER_PATH)
)
print("Silver saved to:", SILVER_PATH)

